# 經文對齊（M2）：開發集檢查（需要 GPU）

候選功能：房主在廣播房間貼上本場綱要後，講員**照恢復本讀經**時，字幕換成恢復本原文並標出經節；不確定就不換。

這本筆記本會：
1. 從雲端硬碟的 `twgbr-archive` 建立**恢復本朗讀開發集** `bible-dev-v1`（20 章，錄音＋經文；已有就沿用）；
2. 用和網站相同的 Whisper large-v3-turbo、10 秒停頓切段辨識（Colab T4 GPU，約 15–30 分鐘；已辨識過會沿用）；
3. 計分：恢復本朗讀（讀經時換對多少）＋生命讀經開發集（講解時有沒有亂換；沿用先前的辨識結果）。

**開始前**：「執行階段 → 變更執行階段類型 → T4 GPU」→ 儲存，再「全部執行」。不用 Cloudflare 額度。

跑完回覆「直接讀」。

In [ ]:
# 第 1 格：連接雲端硬碟、找到資料
import os, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
found = [Path(d) for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and 'recovery-version' in dirs and 'life-study' in dirs]
if not found:
    raise RuntimeError('找不到 twgbr-archive（含 recovery-version 與 life-study）')
ARCHIVE = found[0]
BIBLE = ARCHIVE/'bible-dev-v1'
lives = [Path(d)/'life-study-dev-v1' for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and 'life-study-dev-v1' in dirs]
LIFE = lives[0] if lives else None
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() if shutil.which('nvidia-smi') else ''
print('資料：', ARCHIVE, '｜生命讀經開發集：', LIFE, '｜GPU：', gpu or '沒有（請改成 T4 GPU 再執行）')
if not gpu:
    raise RuntimeError('需要 GPU：執行階段 → 變更執行階段類型 → T4 GPU')

In [ ]:
# 第 2 格：取得固定版本的程式與套件（含 GPU 函式庫，約 2–3 分鐘）
import sys
COMMIT = '6da6fd709aedcfa7f78a6cd738dcea62d8cf53ac'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faster-whisper==1.2.1', 'ctranslate2==4.8.3', 'av==16.0.1',
                'rapidfuzz==3.14.1', 'opencc-python-reimplemented==0.1.7', 'nodejs-wheel-binaries==22.20.0', 'beautifulsoup4==4.13.4',
                # CTranslate2 needs CUDA 12 cuBLAS + cuDNN 9; newer Colab images do not ship libcublas.so.12.
                'nvidia-cublas-cu12==12.9.1.4', 'nvidia-cudnn-cu12==9.28.0.42'], check=True)
import importlib, importlib.util
importlib.invalidate_caches()
lib_dirs = [str(Path(location)/'lib') for module in ('nvidia.cublas', 'nvidia.cudnn')
            for location in importlib.util.find_spec(module).submodule_search_locations]
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('CUDA 12 函式庫：', *lib_dirs, sep='\n  ')
import nodejs_wheel
os.environ['NODE_BINARY'] = str(Path(nodejs_wheel.__file__).parent/'bin'/'node')
print('程式版本', COMMIT[:12], '｜Node', subprocess.check_output([os.environ['NODE_BINARY'], '--version'], text=True).strip())


In [ ]:
# 第 3 格：建立恢復本朗讀開發集（20 章；已有就沿用）
if not (BIBLE/'manifest.json').exists():
    result = subprocess.run([sys.executable, str(REPO/'scripts/build_bible_dev_set.py'), '--archive', str(ARCHIVE),
                             '--output', str(BIBLE), '--count', '20'], cwd=REPO, capture_output=True, text=True)
    print(result.stdout[-3000:], result.stderr[-3000:])
    if result.returncode:
        raise RuntimeError('開發集建立失敗，請看上面的訊息')
print('開發集：', BIBLE, sorted(p.name for p in BIBLE.glob('*.mp3'))[:25])

In [ ]:
# 第 4 格：下載固定版本的 Whisper large-v3-turbo 並核對 SHA-256（約 1.6GB）
import hashlib
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download('deepdml/faster-whisper-large-v3-turbo-ct2', revision='4df90f75321148c3a29a9e2351b7ddf8f5b115a8',
                              allow_patterns=['config.json', 'model.bin', 'preprocessor_config.json', 'tokenizer.json', 'vocabulary.json'])
digest = hashlib.sha256()
with open(Path(MODEL_DIR)/'model.bin', 'rb') as stream:
    for block in iter(lambda: stream.read(1 << 24), b''):
        digest.update(block)
if digest.hexdigest() != 'e76620f83d5f5b69efd3d87e3dc180c1bd21df9fbebacfd4335e5e1efcc018da':
    raise RuntimeError('模型檔 SHA-256 不符，已停止')
print('模型已核對：', MODEL_DIR)


In [ ]:
# 第 5 格：辨識＋計分
def latest(folder, pattern):
    runs = sorted(p for p in (folder/'eval-runs').glob('*') if p.is_dir() and any(p.glob(pattern))) if (folder/'eval-runs').is_dir() else []
    return runs[-1] if runs else None
def run(data, extra, reuse=None, model=True, label='verses'):
    out = data/'eval-runs'/(datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + label)
    command = [sys.executable, '-u', str(REPO/'scripts/evaluate_meet.py'), '--meet', str(data), '--output', str(out),
               '--conditions', 'vad10', '--development', '--verses-archive', str(ARCHIVE)] + extra
    if model:
        command += ['--model-dir', MODEL_DIR, '--model-label', 'whisper-large-v3-turbo (deepdml ct2 4df90f7)']
    if reuse:
        command += ['--reuse', str(reuse)]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('評估失敗，請看上面的錯誤訊息')
    print('\n結果：', out/'summary.md')
    return out
BIBLE_RUN = run(BIBLE, [], reuse=latest(BIBLE, '*.whisper_live_vad_10s.json'), label='bible-verses')
if LIFE:
    LIFE_RUN = run(LIFE, ['--loose-reference', '--outline'], reuse=latest(LIFE, '*.whisper_live_vad_10s.json'), label='outline-verses')